In [ ]:
# n = 10
[1,-1,-1,1]
# predicted return, return volatility over prior year, correlation of stock with CRSP value-weighted index (Mkt-RF), OOS R2
# long only strategy cum return - up to 2020
# sharpe ratio is (monthly excess return)/volatility of all dates -> actual; volatility is monthly std over all years
# use logit to get probs for all stocks, select top n highest probs to form 1/n portfolio 

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize, approx_fprime
import torch
import matplotlib.pyplot as plt

from scipy.stats import mstats

In [2]:
link_table = pd.read_csv('permno-comnam.csv', index_col=0)
link_table.set_index('permno', inplace=True)

In [3]:
green_vars = pd.read_csv("green cleaned.csv", dtype={'ncusip': 'string'})
green_vars['ret_fwd_1'] = (green_vars.groupby('permno')['ret_excess'].shift(-1) )
green_vars = green_vars.dropna(subset=["ret_fwd_1"])

green_vars['datadate'] = pd.to_datetime(green_vars['DATE'], format='%Y%m%d')+pd.tseries.offsets.MonthEnd(0)
green_vars = green_vars.set_index('permno', drop=False)
green_vars = green_vars[['datadate', 'mve_m', 'ret_fwd_1']]

In [4]:
print(green_vars)

         datadate         mve_m  ret_fwd_1
permno                                    
10006  1980-01-31  3.034208e+05  -0.067695
10006  1980-02-29  3.676485e+05  -0.184178
10006  1980-03-31  3.410715e+05  -0.040051
10006  1980-04-30  2.823806e+05   0.076577
10006  1980-05-31  2.746290e+05  -0.013676
...           ...           ...        ...
93436  2024-07-31  6.321554e+08  -0.082191
93436  2024-08-31  7.413801e+08   0.217942
93436  2024-09-30  6.840044e+08  -0.048925
93436  2024-10-31  8.390474e+08   0.377469
93436  2024-11-30  8.020335e+08   0.166308

[267678 rows x 3 columns]


In [5]:
green_vars[green_vars['datadate']=='2023-11-30'].sort_values(by='mve_m', ascending=False).head(10).loc[93436]

datadate     2023-11-30 00:00:00
mve_m                638454482.0
ret_fwd_1               0.030688
Name: 93436, dtype: object

## Value weighted

In [9]:
portfolio_returns = []
print_firms=True

for date in sorted(green_vars['datadate'].unique())[-60:-8]:
    # Get data for this date
    date_data = green_vars[green_vars['datadate'] == date].copy()
    
    # Remove rows with missing returns or missing market values
    date_data = date_data.dropna(subset=['ret_fwd_1', 'mve_m'])
    
    if len(date_data) > 0 and date_data['mve_m'].sum() > 0:
        # Calculate weights (market value / total market value)
        weights = date_data['mve_m'] / date_data['mve_m'].sum()

        if print_firms:
            w = weights.to_frame(name='weight')
            merged = w.merge(link_table, on='permno', how='left').sort_values(by='weight',ascending=False).head(10)
            print(date)
            print(merged)

        
        # Calculate value-weighted return
        vw_return = (weights * date_data['ret_fwd_1']).sum()
        
        portfolio_returns.append(vw_return)

# Convert to array
portfolio_returns = np.array(portfolio_returns)

# Calculate Sharpe ratio (assuming monthly returns)
mean_return = portfolio_returns.mean()
std_return = portfolio_returns.std()
sharpe_ratio = mean_return / std_return

# Annualized Sharpe ratio (multiply by sqrt(12) for monthly data)
sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)

print(f"Mean Monthly Return: {mean_return:.4f}")
print(f"Monthly Std Dev: {std_return:.4f}")
print(f"Monthly Sharpe Ratio: {sharpe_ratio:.4f}")
print(f"Annualized Sharpe Ratio: {sharpe_ratio_annualized:.4f}")
print(f"Number of periods: {len(portfolio_returns)}")

2020-01-31 00:00:00
          weight                comnam
permno                                
14593   0.051318             APPLE INC
10107   0.047835        MICROSOFT CORP
84788   0.036675        AMAZON COM INC
13407   0.019690    META PLATFORMS INC
47896   0.017425   JPMORGAN CHASE & CO
22111   0.015301     JOHNSON & JOHNSON
55976   0.013438           WALMART INC
92611   0.012819              VISA INC
59408   0.012626  BANK OF AMERICA CORP
18163   0.012414   PROCTER & GAMBLE CO
2020-02-29 00:00:00
          weight               comnam
permno                               
14593   0.054099            APPLE INC
10107   0.051723       MICROSOFT CORP
84788   0.039946       AMAZON COM INC
13407   0.019404   META PLATFORMS INC
47896   0.016584  JPMORGAN CHASE & CO
22111   0.015652    JOHNSON & JOHNSON
92611   0.013560             VISA INC
55976   0.012976          WALMART INC
91233   0.012571       MASTERCARD INC
18163   0.012294  PROCTER & GAMBLE CO
2020-03-31 00:00:00
          weight

In [13]:
portfolio_returns = []
portfolio_turnover = []
prev_weights_dict = {}
prev_returns_dict = {}
prev_gross_return = 0.0
transaction_cost = 0.001  # 50 bps, same as your nodewise backtest

print_firms = True

for date in sorted(green_vars['datadate'].unique())[-14:-8]:
    # Get data for this date
    date_data = green_vars[green_vars['datadate'] == date].copy()
    
    # Remove rows with missing returns or missing market values
    date_data = date_data.dropna(subset=['ret_fwd_1', 'mve_m'])
    
    if len(date_data) > 0 and date_data['mve_m'].sum() > 0:
        # Calculate weights (market value / total market value)
        weights = date_data['mve_m'] / date_data['mve_m'].sum()
        
        # Create weights dictionary
        weights_dict = weights.to_dict()
        returns_dict = date_data['ret_fwd_1'].to_dict()
        
        if print_firms:
            w = weights.to_frame(name='weight')
            merged = w.merge(link_table, on='permno', how='left').sort_values(by='weight', ascending=False).head(10)
            print(f"\n{date}")
            print(merged)
        
        # Calculate gross value-weighted return
        gross_return = (weights * date_data['ret_fwd_1']).sum()
        
        # Calculate transaction costs
        if len(prev_weights_dict) > 0:
            # Get all assets (current + previous)
            all_assets = set(weights_dict.keys()) | set(prev_weights_dict.keys())
            
            # Adjust previous weights: w+_{t,j} = w_{t,j} * (1 + r_{t,j}) / (1 + r_p,t)
            adjusted_prev = {}
            for asset in all_assets:
                prev_w = prev_weights_dict.get(asset, 0.0)
                
                if asset in prev_returns_dict and prev_gross_return > -0.99:
                    prev_r = prev_returns_dict[asset]
                    adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
                else:
                    adjusted_prev[asset] = 0.0
            
            # Renormalize adjusted weights (only over current assets)
            adj_sum = sum(adjusted_prev.get(a, 0.0) for a in weights_dict.keys())
            if adj_sum > 1e-10:
                adjusted_prev_normalized = {k: adjusted_prev.get(k, 0.0)/adj_sum 
                                           for k in weights_dict.keys()}
            else:
                adjusted_prev_normalized = {k: 0.0 for k in weights_dict.keys()}
            
            # Turnover: sum over all assets
            turnover = sum(abs(weights_dict.get(a, 0.0) - adjusted_prev_normalized.get(a, 0.0)) 
                          for a in all_assets)
            
            # Transaction cost
            tc = transaction_cost * (1 + gross_return) * turnover
        else:
            # First period: full turnover
            turnover = sum(abs(w) for w in weights_dict.values())
            tc = transaction_cost * turnover
        
        # Net return
        net_return = gross_return - tc
        
        portfolio_returns.append(net_return)
        portfolio_turnover.append(turnover)
        
        # Update for next iteration
        prev_weights_dict = weights_dict.copy()
        prev_returns_dict = returns_dict.copy()
        prev_gross_return = gross_return

# Convert to array
portfolio_returns = np.array(portfolio_returns)
portfolio_turnover = np.array(portfolio_turnover)

# Calculate Sharpe ratio
mean_return = portfolio_returns.mean()
std_return = portfolio_returns.std()
sharpe_ratio = mean_return / std_return if std_return > 0 else 0
sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)

print(f"\n{'='*60}")
print("S&P 500 PERFORMANCE (WITH TRANSACTION COSTS)")
print(f"{'='*60}")
print(f"Mean Monthly Return:      {mean_return:.4f}")
print(f"Monthly Std Dev:          {std_return:.4f}")
print(f"Monthly Sharpe Ratio:     {sharpe_ratio:.4f}")
print(f"Annualized Sharpe Ratio:  {sharpe_ratio_annualized:.4f}")
print(f"Average Turnover:         {portfolio_turnover.mean():.4f}")
print(f"Number of periods:        {len(portfolio_returns)}")
print(f"{'='*60}")


2023-11-30 00:00:00
          weight                  comnam
permno                                  
14593   0.081481               APPLE INC
10107   0.077093          MICROSOFT CORP
84788   0.042194          AMAZON COM INC
86580   0.030852             NVIDIA CORP
13407   0.020515      META PLATFORMS INC
93436   0.019587               TESLA INC
50876   0.016132          LILLY ELI & CO
92655   0.015219  UNITEDHEALTH GROUP INC
55976   0.013493             WALMART INC
11850   0.012869        EXXON MOBIL CORP

2023-12-31 00:00:00
          weight                  comnam
permno                                  
14593   0.082900               APPLE INC
10107   0.079025          MICROSOFT CORP
84788   0.042364          AMAZON COM INC
86580   0.032417             NVIDIA CORP
93436   0.021416               TESLA INC
13407   0.020376      META PLATFORMS INC
50876   0.015745          LILLY ELI & CO
92655   0.014352  UNITEDHEALTH GROUP INC
47896   0.012662     JPMORGAN CHASE & CO
55976   0.01175

## Equal weighted

In [7]:
portfolio_returns_ew = []
print_firms = True

for date in sorted(green_vars['datadate'].unique())[-13-264:-13]:
    # Get data for this date
    date_data = green_vars[green_vars['datadate'] == date].copy()
    
    # Remove rows with missing returns or missing market values
    date_data = date_data.dropna(subset=['ret_fwd_1', 'mve_m'])
    
    if len(date_data) > 0:
        # Calculate equal weights (1/N for each firm)
        n_firms = len(date_data)
        weights = pd.Series(1/n_firms, index=date_data.index)
        
        if print_firms:
            w = weights.to_frame(name='weight')
            merged = w.merge(link_table, on='permno', how='left').sort_values(by='weight', ascending=False).head(10)
            print(date)
            print(merged)
        
        # Calculate equal-weighted return
        ew_return = date_data['ret_fwd_1'].mean()
        
        portfolio_returns_ew.append(ew_return)

# Convert to array
portfolio_returns_ew = np.array(portfolio_returns_ew)

# Calculate Sharpe ratio (assuming monthly returns)
mean_return = portfolio_returns_ew.mean()
std_return = portfolio_returns_ew.std()
sharpe_ratio = mean_return / std_return

# Annualized Sharpe ratio (multiply by sqrt(12) for monthly data)
sharpe_ratio_annualized = sharpe_ratio * np.sqrt(12)

print(f"Mean Monthly Return: {mean_return:.4f}")
print(f"Monthly Std Dev: {std_return:.4f}")
print(f"Monthly Sharpe Ratio: {sharpe_ratio:.4f}")
print(f"Annualized Sharpe Ratio: {sharpe_ratio_annualized:.4f}")
print(f"Number of periods: {len(portfolio_returns_ew)}")

2001-12-31 00:00:00
          weight                      comnam
permno                                      
10078   0.002132        SUN MICROSYSTEMS INC
62770   0.002132      AMSOUTH BANCORPORATION
65875   0.002132  VERIZON COMMUNICATIONS INC
65787   0.002132         TRIBUNE COMPANY NEW
65138   0.002132               BANK ONE CORP
64995   0.002132                 KEYCORP NEW
64936   0.002132         DOMINION ENERGY INC
64565   0.002132  COUNTRYWIDE FINANCIAL CORP
64390   0.002132         PROGRESSIVE CORP OH
64311   0.002132       NORFOLK SOUTHERN CORP
2002-01-31 00:00:00
          weight                      comnam
permno                                      
10078   0.002141        SUN MICROSYSTEMS INC
62770   0.002141      AMSOUTH BANCORPORATION
65875   0.002141  VERIZON COMMUNICATIONS INC
65787   0.002141         TRIBUNE COMPANY NEW
65138   0.002141               BANK ONE CORP
64995   0.002141                 KEYCORP NEW
64936   0.002141         DOMINION ENERGY INC
64565   0.00214